Applies anonymisation to Reddit datasets begore submission
Author usernames and post URLs were not used in any analytical procedure

In [1]:
import json
import hashlib
from pathlib import Path
import pandas as pd

In [2]:
# source
RAW_SRC   = Path("reddit_raw")
CLEAN_SRC = Path("reddit_clean")

# anonymised copies for submission
RAW_DST   = Path("reddit_raw_submit")
CLEAN_DST = Path("reddit_clean_submit")

RAW_DST.mkdir(exist_ok=True)
CLEAN_DST.mkdir(exist_ok=True)

print(f"source:      {RAW_SRC}/  and  {CLEAN_SRC}/")
print(f"destination: {RAW_DST}/  and  {CLEAN_DST}/")

source:      reddit_raw/  and  reddit_clean/
destination: reddit_raw_submit/  and  reddit_clean_submit/


In [4]:
def hash_id(pid):
    """SHA-256 hash of post ID, truncated to 12 hex chars. deterministic and one-way."""
    if pid is None or (isinstance(pid, float) and pd.isna(pid)):
        return None
    return hashlib.sha256(str(pid).encode("utf-8")).hexdigest()[:12]

# sanity check
print(f"hash('abc123')  = {hash_id('abc123')}")
print(f"hash('abc123')  = {hash_id('abc123')}   (should match)")
print(f"hash('xyz789')  = {hash_id('xyz789')}   (should differ)")

hash('abc123')  = 6ca13d52ca70
hash('abc123')  = 6ca13d52ca70   (should match)
hash('xyz789')  = 5a4640c17e8e   (should differ)


In [5]:
def anonymise_post_dict(post):
    """Return a new dict with author replaced, url removed, id hashed."""
    out = dict(post)
    if "author" in out:
        out["author"] = "[anonymised]"
    if "url" in out:
        del out["url"]
    if "id" in out and out["id"] is not None:
        out["id"] = hash_id(out["id"])
    return out

n_files = 0
n_records = 0
n_had_author = 0
n_had_url = 0

# JSON
for src_path in RAW_SRC.rglob("*.json"):
    rel = src_path.relative_to(RAW_SRC)
    dst_path = RAW_DST / rel
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(src_path, "r", encoding="utf-8") as f:
        posts = json.load(f)
    
    anon_posts = []
    for p in posts:
        if "author" in p: n_had_author += 1
        if "url" in p:    n_had_url += 1
        anon_posts.append(anonymise_post_dict(p))
        n_records += 1
    
    with open(dst_path, "w", encoding="utf-8") as f:
        json.dump(anon_posts, f, ensure_ascii=False)
    n_files += 1

#CSV
for extra in RAW_SRC.rglob("*.csv"):
    rel = extra.relative_to(RAW_SRC)
    dst_path = RAW_DST / rel
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(extra)
    for col in ["author", "url"]:
        if col in df.columns:
            df = df.drop(columns=[col])
    if "id" in df.columns:
        df["id"] = df["id"].apply(hash_id)
    df.to_csv(dst_path, index=False)

print(f"JSON files processed:        {n_files}")
print(f"Post records processed:      {n_records}")
print(f"Records with 'author' field: {n_had_author}")
print(f"Records with 'url' field:    {n_had_url}")
print(f"\nAnonymised raw archive: {RAW_DST}/")

JSON files processed:        210
Post records processed:      2992
Records with 'author' field: 2992
Records with 'url' field:    2992

Anonymised raw archive: reddit_raw_submit/


In [6]:
for check_path in RAW_DST.rglob("*.json"):
    with open(check_path) as f:
        sample = json.load(f)
    if sample:
        break

print(f"sample file: {check_path}")
print(f"records:     {len(sample)}\n")
print("first record fields:")
for k, v in sample[0].items():
    print(f"  {k:20s} | {str(v)[:80]}")

assert all(p.get("author") == "[anonymised]" for p in sample if "author" in p), "author not anonymised"
assert all("url" not in p for p in sample), "url still present"
assert all(len(p["id"]) == 12 for p in sample if "id" in p and p["id"] is not None), "id not hashed"

sample file: reddit_raw_submit/RoboCare/unitedkingdom__elderly_care.json
records:     4

first record fields:
  author               | [anonymised]
  created_utc          | 1720484902
  id                   | 0b078b8f185e
  link_flair_text      | None
  num_comments         | 0
  score                | 27
  selftext             | 
  subreddit            | unitedkingdom
  title                | Runcorn care home accused of neglect by elderly resident's mother
  _matched_domain      | RoboCare
  _matched_query       | "elderly care"


In [7]:
def anonymise_dataframe(df):
    """Drop author/url columns; hash id column. Returns a new DataFrame."""
    df = df.copy()
    for col in ["author", "url"]:
        if col in df.columns:
            df = df.drop(columns=[col])
    if "id" in df.columns:
        df["id"] = df["id"].apply(hash_id)
    return df

report = []
for src_csv in CLEAN_SRC.rglob("*.csv"):
    rel = src_csv.relative_to(CLEAN_SRC)
    dst_csv = CLEAN_DST / rel
    dst_csv.parent.mkdir(parents=True, exist_ok=True)
    
    df_orig = pd.read_csv(src_csv)
    dropped = [c for c in ["author", "url"] if c in df_orig.columns]
    id_hashed = "id" in df_orig.columns
    
    df_anon = anonymise_dataframe(df_orig)
    df_anon.to_csv(dst_csv, index=False)
    
    report.append({
        "file": str(rel),
        "rows": len(df_orig),
        "dropped_cols": ", ".join(dropped) if dropped else "(none)",
        "id_hashed": id_hashed,
    })

print(pd.DataFrame(report).to_string(index=False))
print(f"\nanonymised cleaned archive: {CLEAN_DST}/")

                         file  rows dropped_cols  id_hashed
             Cancer_clean.csv     2  author, url       True
               _all_clean.csv   188  author, url       True
 _class_alignment_primary.csv     4       (none)      False
                 FR_clean.csv    17  author, url       True
         _alignment_ranks.csv     8       (none)      False
            Chatbot_clean.csv     0  author, url       True
                Car_clean.csv    11  author, url       True
               Loan_clean.csv     4  author, url       True
                LLM_clean.csv   131  author, url       True
        _all_posts_scored.csv   181  author, url       True
   _class_alignment_sens1.csv     4       (none)      False
           RoboCare_clean.csv     1  author, url       True
  _all_clean_preprocessed.csv   188  author, url       True
_domain_sentiment_summary.csv     8       (none)      False
                 WB_clean.csv    22  author, url       True

anonymised cleaned archive: reddit_clea

In [8]:
orig = pd.read_csv(CLEAN_SRC / "_all_posts_scored.csv")
anon = pd.read_csv(CLEAN_DST / "_all_posts_scored.csv")

print(f"original columns  ({len(orig.columns)}): {list(orig.columns)}\n")
print(f"anonymised columns ({len(anon.columns)}): {list(anon.columns)}\n")

print(f"Row count preserved: {len(orig) == len(anon)}  ({len(orig)} rows)")
print(f"'author' removed:    {'author' not in anon.columns}")
print(f"'url' removed:       {'url' not in anon.columns}\n")


analytical_cols = ["compound_mean", "compound_median", "compound_std",
                   "pos_sent_pct", "neg_sent_pct", "neu_sent_pct",
                   "compound_wholepost", "n_sentences",
                   "assigned_domain", "subreddit", "combined_text", "text_for_vader"]
for col in analytical_cols:
    if col in orig.columns and col in anon.columns:
        print(f"  {col:22s} identical: {orig[col].equals(anon[col])}")


sample_orig = orig["id"].iloc[0]
sample_anon = anon["id"].iloc[0]
print(f"\nID hash check:")
print(f"  original:   {sample_orig}")
print(f"  hashed:     {sample_anon}")
print(f"  recomputed: {hash_id(sample_orig)}")
assert hash_id(sample_orig) == sample_anon, "hashing not deterministic"

original columns  (32): ['id', 'subreddit', 'author', 'created_utc', 'score', 'num_comments', 'url', 'title', 'selftext', 'combined_text', 'text_char_len', 'fetch_domain', 'assigned_domain', 'n_core_hits', 'matched_core', 'matched_context', 'passed_uk_filter', 'n_passing_domains', 'text_for_vader', 'vader_char_len', 'vader_word_len', 'orig_char_len', 'char_removed_pct', 'is_news_digest', 'n_sentences', 'compound_mean', 'compound_median', 'compound_std', 'pos_sent_pct', 'neg_sent_pct', 'neu_sent_pct', 'compound_wholepost']

anonymised columns (30): ['id', 'subreddit', 'created_utc', 'score', 'num_comments', 'title', 'selftext', 'combined_text', 'text_char_len', 'fetch_domain', 'assigned_domain', 'n_core_hits', 'matched_core', 'matched_context', 'passed_uk_filter', 'n_passing_domains', 'text_for_vader', 'vader_char_len', 'vader_word_len', 'orig_char_len', 'char_removed_pct', 'is_news_digest', 'n_sentences', 'compound_mean', 'compound_median', 'compound_std', 'pos_sent_pct', 'neg_sent_pct